In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random

import torch.optim as optim
import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset,DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

seed=42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

cpu


In [2]:
data=load_breast_cancer()
x,y=data.data,((data.target == 0).astype(np.float32))
x.shape


(569, 30)

In [3]:
x_train,x_temp,y_train,y_temp=train_test_split(x,y,test_size=0.3,random_state=42,stratify=y)
x_val,x_test,y_val,y_test=train_test_split(x_temp,y_temp,test_size=0.5,random_state=42,stratify=y_temp)
x_val.shape

(85, 30)

In [4]:
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_val_scaled=scaler.transform(x_val)
x_test_scaled=scaler.transform(x_test)

x_train_tensor=torch.tensor(x_train,dtype=torch.float32)
x_val_tensor=torch.tensor(x_val,dtype=torch.float32)
x_test_tensor=torch.tensor(x_test,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.float32).view(-1,1)
y_val_tensor=torch.tensor(y_val,dtype=torch.float32).view(-1,1)
y_test_tensor=torch.tensor(y_test,dtype=torch.float32).view(-1,1)

train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
val_dataset=TensorDataset(x_val_tensor,y_val_tensor)

batch_size=32

train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=batch_size,shuffle=True)

In [21]:
class ANN(nn.Module):

    def __init__(self,input_features,use_bn_dropout=False):
        super().__init__()

        layers=[]
        layers.append(nn.Linear(input_features,128))
        if use_bn_dropout:
            layers.append(nn.BatchNorm1d(128))
        layers.append(nn.ReLU())
        if use_bn_dropout:
            layers.append(nn.Dropout(0.3))

        layers.append(nn.Linear(128,64))
        if use_bn_dropout:
            layers.append(nn.BatchNorm1d(64))
        layers.append(nn.ReLU())
        if use_bn_dropout:
            layers.append(nn.Dropout(0.3))

        layers.append(nn.Linear(64,1))
        self.network=nn.Sequential(*layers)

    def forward(self,x):
        return self.network(x)

class EarlyStopping:

    def __init__(self,patience=10,min_delta=0.001):
        self.patience=patience
        self.min_delta=min_delta

        self.best_loss=float('inf')
        self.counter=0
        self.best_weights=None

    def __call__(self,val_loss,model):

        if val_loss < self.best_loss-self.min_delta:
            self.best_loss=val_loss
            self.counter=0
            self.best_weights=copy.deepcopy(model.state_dict())

        else:
            self.counter+=1

        if self.counter >= self.patience:
            return True
        return False

In [22]:

def train_model(use_bn_dropout):

    model = ANN(
        input_features=x_train.shape[1],
        use_bn_dropout=use_bn_dropout
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    early_stopping = EarlyStopping(patience=10)

    train_loss = []
    val_loss = []

    epochs = 100

    for epoch in range(epochs):

        # =====================
        # TRAINING
        # =====================

        model.train()

        running_train_loss = 0

        for x_batch, y_batch in train_loader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(x_batch)

            loss = criterion(outputs, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()

        avg_train_loss = running_train_loss / len(train_loader)

        # =====================
        # VALIDATION
        # =====================

        model.eval()

        running_val_loss = 0

        with torch.no_grad():

            for x_batch, y_batch in val_loader:

                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                outputs = model(x_batch)

                loss = criterion(outputs, y_batch)

                running_val_loss += loss.item()

        avg_val_loss = running_val_loss / len(val_loader)

        # Store losses
        train_loss.append(avg_train_loss)
        val_loss.append(avg_val_loss)

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {avg_train_loss:.4f} "
            f"Val Loss: {avg_val_loss:.4f}"
        )

        # =====================
        # EARLY STOPPING
        # =====================

        if early_stopping(avg_val_loss, model):

            print("Early stopping triggered!")
            break

    # =====================
    # RESTORE BEST WEIGHTS
    # =====================

    if early_stopping.best_weights is not None:

        model.load_state_dict(
            early_stopping.best_weights
        )

    return model, train_loss, val_loss


# Train Model A
print("\nTraining Model A: Normal ANN")

model_a, train_a, val_a = train_model(False)


# Train Model B
print("\nTraining Model B: BatchNorm + Dropout")

model_b, train_b, val_b = train_model(True)



Training Model A: Normal ANN
Epoch [1/100] Train Loss: 2.6890 Val Loss: 1.9235
Epoch [2/100] Train Loss: 0.9026 Val Loss: 1.0314
Epoch [3/100] Train Loss: 0.4118 Val Loss: 0.5319
Epoch [4/100] Train Loss: 0.3328 Val Loss: 0.3738
Epoch [5/100] Train Loss: 0.3191 Val Loss: 0.3016
Epoch [6/100] Train Loss: 0.2443 Val Loss: 0.2994
Epoch [7/100] Train Loss: 0.2479 Val Loss: 0.5977
Epoch [8/100] Train Loss: 0.2785 Val Loss: 0.3123
Epoch [9/100] Train Loss: 0.3195 Val Loss: 0.2904
Epoch [10/100] Train Loss: 0.2437 Val Loss: 0.2917
Epoch [11/100] Train Loss: 0.2590 Val Loss: 0.2811
Epoch [12/100] Train Loss: 0.2547 Val Loss: 0.2856
Epoch [13/100] Train Loss: 0.2349 Val Loss: 0.3293
Epoch [14/100] Train Loss: 0.2563 Val Loss: 0.3381
Epoch [15/100] Train Loss: 0.2235 Val Loss: 0.2537
Epoch [16/100] Train Loss: 0.2085 Val Loss: 0.2863
Epoch [17/100] Train Loss: 0.2297 Val Loss: 0.3189
Epoch [18/100] Train Loss: 0.2899 Val Loss: 0.5037
Epoch [19/100] Train Loss: 0.2423 Val Loss: 0.2633
Epoch [20/

In [14]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)
print("torch path:", torch.__file__)
print("BatchNorm1d:", nn.BatchNorm1d)

PyTorch version: 2.14.1+cpu
torch path: c:\Users\mahad\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\__init__.py
BatchNorm1d: <class 'torch.nn.modules.batchnorm.BatchNorm1d'>


In [30]:

def evaluate_model(model):

    model.eval()

    # Convert NumPy array to PyTorch tensor
    x_test_tensor = torch.tensor(
        x_test,
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():

        logits = model(x_test_tensor)

        probabilities = torch.sigmoid(logits)

        predictions = (
            probabilities >= 0.5
        ).float()

    y_true = np.asarray(y_test).flatten()

    y_pred = predictions.cpu().numpy().flatten()

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )

    recall = recall_score(
        y_true, y_pred, zero_division=0
    )

    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )
    model_name='model_1'

    print(f"\n{model_name} RESULTS")

    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1 Score:", round(f1, 4))

    print("\nClassification Report:")

    print(classification_report(
        y_true,
        y_pred,
        target_names=["Benign", "Malignant"],
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred)

    print("Confusion Matrix:\n", cm)

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }


In [31]:
print(evaluate_model(model_a))


model_1 RESULTS
Accuracy: 0.9535
Precision: 0.9667
Recall: 0.9062
F1 Score: 0.9355

Classification Report:
              precision    recall  f1-score   support

      Benign       0.95      0.98      0.96        54
   Malignant       0.97      0.91      0.94        32

    accuracy                           0.95        86
   macro avg       0.96      0.94      0.95        86
weighted avg       0.95      0.95      0.95        86

Confusion Matrix:
 [[53  1]
 [ 3 29]]
{'Model': 'model_1', 'Accuracy': 0.9534883720930233, 'Precision': 0.9666666666666667, 'Recall': 0.90625, 'F1 Score': 0.9354838709677419}
